# Task 9: Freeze the October analysis sample and rank columns

Start with `merged_v0.csv`: one row per six-digit 2018 SOC occupation. Keep the same occupations for every headline comparison: those with AIOE, Frey–Osborne, Eloundou GPT-4 beta, and Eloundou human beta scores.

Ranks are computed **within this fixed sample**, separately for each score. Higher values mean greater exposure or computerisation risk. Use ascending average ranks divided by the sample size: ties share their average position, and ranks are on a 0–1 scale. These are relative positions, not probabilities of job loss.

Keep all original columns and raw scores. Missing O*NET features do not affect eligibility. No missing scores are imputed.

Run from the `notebooks/` folder or the repository root. Requires pandas from `requirements.txt`. Exports the sample, an audit of every input occupation, and a methods note to `data/datasets/processed/`.

In [1]:
from pathlib import Path
import hashlib
import math
import pandas as pd

processed_folder = Path('../data/datasets/processed')
if not processed_folder.is_dir():
    processed_folder = Path('data/datasets/processed')
input_path = processed_folder / 'merged_v0.csv'
input_bytes = input_path.read_bytes()
source_sha256 = hashlib.sha256(input_bytes).hexdigest()

master = pd.read_csv(input_path, dtype={'soc2018': 'string'})
score_to_rank = {
    'aioe': 'rank_aioe',
    'fo_prob': 'rank_fo',
    'dv_rating_beta': 'rank_elo_dv',
    'human_rating_beta': 'rank_elo_hum',
}
score_columns = list(score_to_rank)

assert {'soc2018', 'title', *score_columns}.issubset(master.columns)
assert master['soc2018'].notna().all(), 'Missing SOC codes'
assert master['soc2018'].is_unique, 'Duplicate SOC codes'
assert master['soc2018'].str.fullmatch(r'\d{2}-\d{4}').all(), 'Invalid SOC format'
assert len(master) == 798, f'Expected 798 input occupations; found {len(master)}'

for column in score_columns:
    values = pd.to_numeric(master[column], errors='raise')
    assert values.dropna().map(math.isfinite).all(), f'Nonfinite values in {column}'
    if column != 'aioe':
        assert values.dropna().between(0, 1).all(), f'Out-of-range scores in {column}'

print(f'Input: {len(master)} occupations, {len(master.columns)} columns; SOC keys verified.')
print(master[score_columns].notna().sum().to_string())

Input: 798 occupations, 214 columns; SOC keys verified.
aioe                 790
fo_prob              663
dv_rating_beta       798
human_rating_beta    798


## 1. Select the fixed sample and explain exclusions

Eligibility depends only on the four headline scores. Record a missing-value flag for each score, a missing-score count, an inclusion flag, and the names of every missing score. The audit retains all 798 occupations; the analysis sample retains only eligible occupations.

The expected sample is 663 occupations. A changed count stops execution before export so the team can investigate rather than silently redefine the sample.

In [2]:
missing = master[score_columns].isna()
audited = master.copy()
missing_flag_columns = []
for column in score_columns:
    flag = 'missing_' + column
    audited[flag] = missing[column]
    missing_flag_columns.append(flag)

audited['missing_index_count'] = missing.sum(axis=1)
audited['included_in_analysis'] = ~missing.any(axis=1)
audited['exclusion_reason'] = missing.apply(
    lambda row: '; '.join('missing_' + column for column in score_columns if row[column]),
    axis=1,
)
audit_columns = missing_flag_columns + [
    'missing_index_count', 'included_in_analysis', 'exclusion_reason'
]
audit = audited[['soc2018', 'title'] + score_columns + audit_columns].copy()
sample = audited.loc[audited['included_in_analysis']].copy()

assert len(sample) == 663, f'Expected 663 complete occupations; found {len(sample)}'
assert sample[score_columns].notna().all().all()
assert sample['soc2018'].is_unique
assert sample['exclusion_reason'].eq('').all()
assert sample['missing_index_count'].eq(0).all()
assert audit['included_in_analysis'].eq(audit['missing_index_count'].eq(0)).all()
assert audit.loc[~audit['included_in_analysis'], 'exclusion_reason'].ne('').all()
assert len(sample) + (~audit['included_in_analysis']).sum() == len(master)

print(f'Included: {len(sample)} | Excluded: {len(master) - len(sample)}')
print(audit.loc[~audit['included_in_analysis'], 'exclusion_reason'].value_counts().to_string())

Included: 663 | Excluded: 135
exclusion_reason
missing_fo_prob                  127
missing_aioe; missing_fo_prob      8


## 2. Add percentile ranks and check their meaning

For each of the four scores, `rank(method='average', ascending=True, pct=True)` computes average position divided by 663. A rank near 0.80 means approximately the 80th percentile within this sample. The smallest rank need not be zero, and the largest need not be one when the top score is tied.

All four raw measures already run from lower to higher exposure or computerisation risk, so none is reversed. Alpha and gamma scores remain unchanged for later sensitivity work; they do not determine this sample or receive headline rank columns here.

In [3]:
# A small known-answer check covers ties, zero scores, and missing values.
example = pd.Series([0.0, 0.2, 0.2, 0.9, float('nan')])
expected_ranks = pd.Series([0.25, 0.625, 0.625, 1.0, float('nan')])
pd.testing.assert_series_equal(
    example.rank(method='average', ascending=True, pct=True), expected_ranks
)

rank_columns = list(score_to_rank.values())
for score, rank in score_to_rank.items():
    sample[rank] = sample[score].rank(method='average', ascending=True, pct=True)
    assert sample[rank].notna().all()
    assert sample[rank].gt(0).all() and sample[rank].le(1).all()
    ranks_by_score = sample.groupby(score)[rank]
    assert ranks_by_score.nunique().eq(1).all(), f'Ties differ in {rank}'
    assert ranks_by_score.first().sort_index().is_monotonic_increasing

# Every original value, including O*NET gaps and provenance, is preserved.
pd.testing.assert_frame_equal(
    sample[master.columns], master.loc[audited['included_in_analysis']]
)
print(sample[rank_columns].agg(['min', 'max']).to_string())
print('Checks passed: sample size, score preservation, rank direction, and ties.')

     rank_aioe   rank_fo  rank_elo_dv  rank_elo_hum
min   0.001508  0.001508     0.034691      0.041478
max   1.000000  0.992459     1.000000      0.999246
Checks passed: sample size, score preservation, rank direction, and ties.


## 3. Save the sample, full audit, and methods note

The audit is separate because excluded occupations cannot appear in a complete-case analysis table. Included rows also carry the audit fields so their eligibility is explicit. The methods note records the input fingerprint, exact rank formula, column meanings, exclusions, and limitations.

Rerunning replaces these Task 9 outputs from the current input. The upstream master table is never edited.

In [4]:
sample_path = processed_folder / 'agreement_analysis_sample.csv'
audit_path = processed_folder / 'agreement_sample_audit.csv'
methods_path = processed_folder / 'agreement_sample_definition.md'

excluded = audit.loc[~audit['included_in_analysis']]
exclusion_lines = '\n'.join(
    f'| {reason} | {count} |'
    for reason, count in excluded['exclusion_reason'].value_counts().items()
)
rank_lines = '\n'.join(
    f'| `{score}` | `{rank}` |'
    for score, rank in score_to_rank.items()
)
methods = f'''# Task 9 — October agreement sample definition

## Input and fixed comparison group

Input: `merged_v0.csv`, {len(master)} occupations × {len(master.columns)} columns.
Its SHA-256 fingerprint is `{source_sha256}`.

One row per unique, nonmissing six-digit 2018 SOC code (`soc2018`).
Include an occupation only when all four scores are present: `aioe`, `fo_prob`,
`dv_rating_beta` (GPT-4 beta), and `human_rating_beta` (human beta).
The same {len(sample)} occupations must be used for both label-source comparisons.
No missing score is imputed. O*NET feature completeness is not an eligibility rule.

The sample has {len(sample)} occupations × {len(sample.columns)} columns.
All {len(master.columns)} original columns and their values are retained, including
raw scores, alpha/gamma variants, O*NET features, and crosswalk provenance flags.

## Percentile ranks

Compute each rank within the fixed {len(sample)}-occupation sample, after filtering.
Use ascending average ranks divided by {len(sample)}:
`Series.rank(method='average', ascending=True, pct=True)`.
Ranks lie above zero and at or below one. Tied scores receive the same average
position; tied maxima can have ranks below one. Do not force endpoints to 0 or 1.
For example, `[0, 0.2, 0.2, 0.9]` gives `[0.25, 0.625, 0.625, 1.0]`.

| Preserved raw score | Added rank column |
| --- | --- |
{rank_lines}

All four source scores already increase with exposure or computerisation risk;
none is reversed. AIOE may be negative because it is standardized, not because
its direction is reversed. Alpha/gamma variants are preserved as raw scores;
they do not affect eligibility or receive rank columns in this task.
These are unweighted occupation ranks, not employment-weighted ranks.

## Inclusion audit

`agreement_sample_audit.csv` contains all {len(audit)} input occupations,
identifiers/titles, the four raw scores, and these fields:

- `missing_aioe`, `missing_fo_prob`, `missing_dv_rating_beta`,
  `missing_human_rating_beta`: True when that score is missing.
- `missing_index_count`: number of missing required scores.
- `included_in_analysis`: True only when all four required scores are present.
- `exclusion_reason`: semicolon-separated missing-score flags; blank for included rows.

The same audit fields are also saved on included rows in the sample.
Empty score cells mean missing data, not zero. Empty exclusion reasons mean eligible.

Included: {len(sample)}. Excluded: {len(excluded)}.

| Exclusion reason | Occupations |
| --- | --- |
{exclusion_lines}

## Reproduction and checks

Run `notebooks/task9.ipynb` from top to bottom, with pandas installed from
`requirements.txt`, from either the notebook directory or repository root.
Outputs are `data/datasets/processed/agreement_analysis_sample.csv`,
`agreement_sample_audit.csv`, and this note.
Reload CSV occupation codes as text: `dtype={{'soc2018': 'string'}}`.

The notebook checks required columns, valid/unique SOC keys, numeric and finite
scores, valid probability/share ranges, 798 input occupations, and exactly 663
complete occupations before exporting. It checks rank bounds, direction, equal
ranks for ties, unchanged original values, audit consistency, a known-answer
ranking example, and CSV read-back contents. A changed input/sample count stops
export and requires investigation; do not bypass the check to change the sample.
Reruns regenerate Task 9 outputs; the fingerprint above identifies the input used.
The upstream `merged_v0.csv` is not modified.

## Interpretation and limitations

A rank near 0.80 means roughly the 80th percentile among these 663 occupations;
it is not an 80% probability of automation or job loss. Rankings compare relative
position, not the size of raw-score differences or equivalent constructs.
The indices measure different concepts, and their original vintage/crosswalk
limitations remain. Missing-score exclusions may affect how representative this
sample is; findings apply to the occupations with complete coverage.
This task prepares the sample; correlations, consensus tiers, and disagreement
analysis are later tasks.
'''

# Verify the input has not changed during this run before saving outputs.
assert input_path.read_bytes() == input_bytes, 'Input changed during execution'
sample.to_csv(sample_path, index=False)
audit.to_csv(audit_path, index=False)
methods_path.write_text(methods, encoding='utf-8')

saved_sample = pd.read_csv(sample_path, dtype={'soc2018': 'string'})
saved_audit = pd.read_csv(audit_path, dtype={'soc2018': 'string'})
for original, saved in [(sample, saved_sample), (audit, saved_audit)]:
    expected = original.reset_index(drop=True).copy()
    expected['exclusion_reason'] = expected['exclusion_reason'].where(expected['exclusion_reason'].ne(''), float('nan'))
    pd.testing.assert_frame_equal(expected, saved, check_dtype=False)
assert hashlib.sha256(input_path.read_bytes()).hexdigest() == source_sha256

print(f'Saved sample: {saved_sample.shape[0]} rows × {saved_sample.shape[1]} columns')
print(f'Saved audit: {len(saved_audit)} occupations')
print(f'Methods: {methods_path.name}')
print('All Task 9 checks passed, including exported-file read-back.')

Saved sample: 663 rows × 225 columns
Saved audit: 798 occupations
Methods: agreement_sample_definition.md
All Task 9 checks passed, including exported-file read-back.
